In [12]:
import pandas as pd
import geopandas as gpd
import numpy as np
from unidecode import unidecode
import plotly.graph_objects as go
import json
from pathlib import Path
import webbrowser
import joblib

CSV_PATH = "../outputs/prediksi_2024.csv"
SHP_PATH = "../maps/BATAS KABUPATEN KOTA DESEMBER 2019 DUKCAPIL.shp"
OUT_HTML = "../outputs/dashboard_peta_perbandingan_prediksi.html"
MODEL_PATH = "../models/lasso_model.pkl"

In [13]:
# ----------------------------------------------
# Normalisasi nama wilayah
# ----------------------------------------------
def norm(s):
    if pd.isna(s): return s
    s = unidecode(str(s)).upper().strip()
    s = s.replace("-", " ").replace("/", " ")
    for w in ["KABUPATEN ", "KOTA ", "KAB. ", "ADM. ", "DAERAH ", "ISTIMEWA "]:
        s = s.replace(w, "")
    return " ".join(s.split())

In [15]:
# ----------------------------------------------
# Load CSV
# ----------------------------------------------
df = pd.read_csv(CSV_PATH)

req = {"Wilayah", "P1"}
if not req.issubset(df.columns):
    raise ValueError(f"CSV harus punya kolom {req}")

# ----------------------------------------------
# Load Model
# ----------------------------------------------
print("[INFO] Loading model...")
model = joblib.load(MODEL_PATH)
print("[OK] Model loaded")

# ----------------------------------------------
# Feature columns
# ----------------------------------------------
feature_cols = ["UHH", "HLS", "RLS", "Pengeluaran", "TPT", "Kepadatan"]

missing = [c for c in feature_cols if c not in df.columns]
if missing:
    raise ValueError(f"CSV tidak memiliki fitur: {missing}")

# ----------------------------------------------
# Prediksi LASSO
# ----------------------------------------------
print("[INFO] Predicting...")
df["prediksi"] = model.predict(df[feature_cols])
print("[OK] Prediction done")

# ----------------------------------------------
# Join key
# ----------------------------------------------
df["join_key"] = df["Wilayah"].apply(norm)

[INFO] Loading model...
[OK] Model loaded
[INFO] Predicting...
[OK] Prediction done


In [16]:

# ----------------------------------------------
# Load Shapefile
# ----------------------------------------------
gdf = gpd.read_file(SHP_PATH)

prov_cols = [c for c in ["PROVINSI","WADMPR","PROPINSI","PROVNAME","NAME_1","provinsi","Provinsi"] if c in gdf.columns]
if prov_cols:
    pc = prov_cols[0]
    gdf["_prov"] = gdf[pc].apply(norm)
    gdf = gdf[gdf["_prov"] == "JAWA TENGAH"].copy()

# Deteksi kolom nama wilayah
candidates = ["NAMOBJ","WADMKC","WADMKK","KABUPATEN","KAB_KOTA","KABKOT","NAMA_KAB","NAME_2","KABKO","WADMKAB"]
name_col = next((c for c in candidates if c in gdf.columns), None)
if name_col is None:
    obj_cols = [c for c in gdf.columns if gdf[c].dtype=='object']
    csv_keys = set(df["join_key"])
    best,score = None,-1
    for c in obj_cols:
        cand = set(gdf[c].astype(str).apply(norm))
        sc = len(cand & csv_keys)
        if sc > score:
            best,score = c,sc
    name_col = best if best else obj_cols[0]

print(f"[INFO] Kolom nama wilayah: {name_col}")

gdf["join_key"] = gdf[name_col].apply(norm)

# ----------------------------------------------
# Merge
# ----------------------------------------------
merged = gdf.merge(df[["join_key","P1","prediksi"]], on="join_key", how="left")

[INFO] Kolom nama wilayah: KAB_KOTA


In [6]:
# ----------------------------------------------
# Load Shapefile
# ----------------------------------------------
gdf = gpd.read_file(SHP_PATH)

# Filter Jawa Tengah
prov_cols = [c for c in ["PROVINSI","WADMPR","PROPINSI","PROVNAME","NAME_1","provinsi","Provinsi"] if c in gdf.columns]
if prov_cols:
    pc = prov_cols[0]
    gdf["_prov"] = gdf[pc].apply(norm)
    gdf = gdf[gdf["_prov"] == "JAWA TENGAH"].copy()

# Deteksi nama wilayah
candidates = ["NAMOBJ","WADMKC","WADMKK","KABUPATEN","KAB_KOTA","KABKOT","NAMA_KAB","NAME_2","KABKO","WADMKAB"]
name_col = next((c for c in candidates if c in gdf.columns), None)
if name_col is None:
    obj_cols = [c for c in gdf.columns if gdf[c].dtype=='object']
    csv_keys = set(df["join_key"])
    best,score = None,-1
    for c in obj_cols:
        cand = set(gdf[c].astype(str).apply(norm))
        sc = len(cand & csv_keys)
        if sc > score:
            best,score = c,sc
    name_col = best if best else obj_cols[0]

print(f"[INFO] Kolom nama wilayah: {name_col}")

gdf["join_key"] = gdf[name_col].apply(norm)

# ----------------------------------------------
# Merge
# ----------------------------------------------
merged = gdf.merge(df[["join_key","P1","prediksi"]], on="join_key", how="left")


[INFO] Kolom nama wilayah: KAB_KOTA


In [17]:
# ----------------------------------------------
# CRS ke WGS84
# ----------------------------------------------
try:
    if merged.crs is None or merged.crs.to_epsg() != 4326:
        merged = merged.to_crs(4326)
except:
    pass

# ----------------------------------------------
# Simplify geometry (untuk kecilkan HTML)
# ----------------------------------------------
print("[INFO] Simplifying geometry...")
merged["geometry"] = merged["geometry"].simplify(
    tolerance=0.005,
    preserve_topology=True
)

# ----------------------------------------------
# Hitung ERROR
# ----------------------------------------------
merged["error"] = (merged["P1"] - merged["prediksi"]).abs()

# ----------------------------------------------
# Hitung MAE & RMSE (global)
# ----------------------------------------------
mae = float(merged["error"].mean())
rmse = float(np.sqrt(((merged["P1"] - merged["prediksi"])**2).mean()))

[INFO] Simplifying geometry...


In [18]:
# ----------------------------------------------
# Statistik P1
# ----------------------------------------------
mean_p1 = float(merged["P1"].mean())
max_p1_val = float(merged["P1"].max())
min_p1_val = float(merged["P1"].min())
max_p1_row = merged.loc[merged["P1"].idxmax()]
min_p1_row = merged.loc[merged["P1"].idxmin()]

# ----------------------------------------------
# Statistik Prediksi
# ----------------------------------------------
mean_pred = float(merged["prediksi"].mean())
max_pred_val = float(merged["prediksi"].max())
min_pred_val = float(merged["prediksi"].min())
max_pred_row = merged.loc[merged["prediksi"].idxmax()]
min_pred_row = merged.loc[merged["prediksi"].idxmin()]

# ----------------------------------------------
# Statistik Error
# ----------------------------------------------
mean_err = float(merged["error"].mean())
max_err_val = float(merged["error"].max())
min_err_val = float(merged["error"].min())
max_err_row = merged.loc[merged["error"].idxmax()]
min_err_row = merged.loc[merged["error"].idxmin()]

# ----------------------------------------------
# Range warna
# ----------------------------------------------
vmin = float(min(merged["P1"].min(), merged["prediksi"].min()))
vmax = float(max(merged["P1"].max(), merged["prediksi"].max()))

err_min = float(merged["error"].min())
err_max = float(merged["error"].max())

# ----------------------------------------------
# GeoJSON
# ----------------------------------------------
geojson = json.loads(merged.to_json())

# ----------------------------------------------
# Data dasar
# ----------------------------------------------
locations = merged["join_key"]
custom = np.array(merged[name_col].fillna("-"))

In [ ]:
# ----------------------------------------------
# Hover templates
# ----------------------------------------------
hover_p1 = (
    "<b>%{customdata[0]}</b><br>"
    "P1: %{z:.2f}%<br>"
    f"Rata-rata: {mean_p1:.2f}%"
)

hover_pred = (
    "<b>%{customdata[0]}</b><br>"
    "Prediksi: %{z:.2f}%<br>"
    f"Rata-rata: {mean_pred:.2f}%"
)

hover_err = (
    "<b>%{customdata[0]}</b><br>"
    "Error: %{z:.2f}%<br>"
    f"Rata-rata Error: {mean_err:.2f}%"
)

In [19]:
# ----------------------------------------------
# Trace P1
# ----------------------------------------------
trace_p1 = go.Choropleth(
    geojson=geojson,
    featureidkey="properties.join_key",
    locations=locations,
    z=merged["P1"],
    colorscale="OrRd",
    zmin=vmin, zmax=vmax,
    marker_line_color="white",
    marker_line_width=0.5,
    colorbar_title="P1 (%)",
    hovertemplate=hover_p1,
    customdata=np.c_[custom],
    visible=True
)

# ----------------------------------------------
# Trace Prediksi
# ----------------------------------------------
trace_pred = go.Choropleth(
    geojson=geojson,
    featureidkey="properties.join_key",
    locations=locations,
    z=merged["prediksi"],
    colorscale="OrRd",
    zmin=vmin, zmax=vmax,
    marker_line_color="white",
    marker_line_width=0.5,
    colorbar_title="Prediksi (%)",
    hovertemplate=hover_pred,
    customdata=np.c_[custom],
    visible=False
)

# ----------------------------------------------
# Trace Error
# ----------------------------------------------
trace_err = go.Choropleth(
    geojson=geojson,
    featureidkey="properties.join_key",
    locations=locations,
    z=merged["error"],
    colorscale="Reds",
    zmin=err_min, zmax=err_max,
    marker_line_color="white",
    marker_line_width=0.5,
    colorbar_title="Error (%)",
    hovertemplate=hover_err,
    customdata=np.c_[custom],
    visible=False
)

# ----------------------------------------------
# Subtitle
# ----------------------------------------------
title_base = "Indeks Kedalaman Kemiskinan"

subtitle_p1 = (
    f"P1 • Mean: {mean_p1:.2f}% • "
    f"Max: {max_p1_row[name_col]} ({max_p1_val:.2f}%) • "
    f"Min: {min_p1_row[name_col]} ({min_p1_val:.2f}%) • "
    f"MAE: {mae:.3f} • RMSE: {rmse:.3f}"
)

subtitle_pred = (
    f"Prediksi • Mean: {mean_pred:.2f}% • "
    f"Max: {max_pred_row[name_col]} ({max_pred_val:.2f}%) • "
    f"Min: {min_pred_row[name_col]} ({min_pred_val:.2f}%) • "
    f"MAE: {mae:.3f} • RMSE: {rmse:.3f}"
)

subtitle_err = (
    f"Error • Mean: {mean_err:.2f}% • "
    f"Max: {max_err_row[name_col]} ({max_err_val:.2f}%) • "
    f"Min: {min_err_row[name_col]} ({min_err_val:.2f}%) • "
    f"MAE: {mae:.3f} • RMSE: {rmse:.3f}"
)


In [20]:
# ----------------------------------------------
# Figure
# ----------------------------------------------
fig = go.Figure(data=[trace_p1, trace_pred, trace_err])

fig.update_layout(
    title=dict(
        text=f"<b>{title_base}</b><br><span style='font-size:12px'>{subtitle_p1}</span>",
        x=0.5
    ),
    margin=dict(l=40, r=40, t=80, b=40),
    geo=dict(
        fitbounds="locations",
        visible=False,
        projection_type="mercator"
    ),
    updatemenus=[
        dict(
            buttons=[
                dict(
                    label="P1 (Aktual)",
                    method="update",
                    args=[
                        {"visible":[True, False, False]},
                        {"title":{"text":f"<b>{title_base}</b><br><span style='font-size:12px'>{subtitle_p1}</span>"}}
                    ]
                ),
                dict(
                    label="Prediksi (LASSO)",
                    method="update",
                    args=[
                        {"visible":[False, True, False]},
                        {"title":{"text":f"<b>{title_base}</b><br><span style='font-size:12px'>{subtitle_pred}</span>"}}
                    ]
                ),
                dict(
                    label="Error (|P1 - Prediksi|)",
                    method="update",
                    args=[
                        {"visible":[False, False, True]},
                        {"title":{"text":f"<b>{title_base}</b><br><span style='font-size:12px'>{subtitle_err}</span>"}}
                    ]
                )
            ],
            direction="down",
            x=0.05, xanchor="left",
            y=0.95, yanchor="top"
        )
    ],
    annotations=[
        dict(
            x=0.5, y=-0.05, xref="paper", yref="paper",
            text="Sumber: BPS Jawa Tengah (2024) • Model: LASSO Regression",
            showarrow=False,
            font=dict(size=11, color="#555")
        )
    ]
)

# ----------------------------------------------
# Save HTML 
# ----------------------------------------------
fig.write_html(
    OUT_HTML,
    include_plotlyjs="cdn",
    full_html=True
)

print(f"[OK] Dashboard saved: {OUT_HTML}")
webbrowser.open(Path(OUT_HTML).resolve().as_uri())

[OK] Dashboard saved: ../outputs/dashboard_peta_perbandingan_prediksi.html


True